# Day 23 Tutorial：Scaffold-aware OOF Stacking

> **课程附带教程，不是学习者实验记录。** 输入是 ESOL ECFP，目标是 logS；不代表粘合剂模型。

## Goal

从 ESOL SMILES 构造 Bemis–Murcko scaffold groups，用 `GroupKFold` 的显式 split 列表训练 `StackingRegressor`，比较冻结随机森林、冻结 MLP、简单平均和 OOF stacking。


## Setup

外部 scaffold valid 保持独立。树模型读取 Day 07 冻结配置；MLP 固定为 `(32,)`、`alpha=0.001`、早停、`max_iter=300`、`n_iter_no_change=10`。环系为空的分子统一记为 `__ACYCLIC__`，所以不会被随机拆散。


In [1]:
from pathlib import Path
import contextlib
import io

from rdkit import RDLogger
RDLogger.DisableLog("rdApp.warning")

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    import deepchem as dc

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "public" / "esol.md").exists():
            return candidate
    raise RuntimeError("请从 ML-practice 仓库内运行本教程。")

REPO_ROOT = find_repo_root()
CACHE_DIR = REPO_ROOT / ".cache" / "deepchem"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

featurizer = dc.feat.CircularFingerprint(size=1024, radius=2)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    tasks, datasets, transformers = dc.molnet.load_delaney(
        featurizer=featurizer,
        splitter="scaffold",
        transformers=[],
        reload=True,
        data_dir=str(CACHE_DIR),
        save_dir=str(CACHE_DIR),
    )

train_dataset, valid_dataset, _sealed_test_dataset = datasets
X_train = np.asarray(train_dataset.X)
y_train = np.asarray(train_dataset.y).reshape(-1)
X_valid = np.asarray(valid_dataset.X)
y_valid = np.asarray(valid_dataset.y).reshape(-1)
train_ids = np.asarray(train_dataset.ids).astype(str)
valid_ids = np.asarray(valid_dataset.ids).astype(str)

assert transformers == []
assert X_train.shape == (902, 1024) and y_train.shape == (902,)
assert X_valid.shape == (113, 1024) and y_valid.shape == (113,)
print("Task:", tasks[0])
print("Train / valid:", X_train.shape, X_valid.shape)
print("测试集对象保持封存，本教程不创建测试预测。")


Task: measured log solubility in mols per litre
Train / valid: (902, 1024) (113, 1024)
测试集对象保持封存，本教程不创建测试预测。


In [2]:
import json
import warnings
from time import perf_counter

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def murcko_group(smiles):
    molecule = Chem.MolFromSmiles(str(smiles))
    if molecule is None:
        raise ValueError(f"无法解析 SMILES：{smiles}")
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(
        mol=molecule, includeChirality=False
    )
    return scaffold or "__ACYCLIC__"

train_scaffolds = np.asarray([murcko_group(item) for item in train_ids])
valid_scaffolds = np.asarray([murcko_group(item) for item in valid_ids])
assert set(train_scaffolds).isdisjoint(set(valid_scaffolds))

inner_splits = list(
    GroupKFold(n_splits=5).split(
        X_train, y_train, groups=train_scaffolds
    )
)
coverage = np.zeros(len(y_train), dtype=int)
for fit_idx, hold_idx in inner_splits:
    assert set(train_scaffolds[fit_idx]).isdisjoint(
        set(train_scaffolds[hold_idx])
    )
    coverage[hold_idx] += 1
assert np.all(coverage == 1)

config_path = (
    REPO_ROOT
    / "experiments"
    / "esol"
    / "day01_baseline"
    / "results"
    / "run_config.json"
)
if not config_path.exists():
    raise FileNotFoundError(
        "缺少 Day 07 冻结配置，无法复现树模型：" + str(config_path)
    )
frozen = json.loads(config_path.read_text(encoding="utf-8"))
rf_params = frozen["model_params"]["random_forest"]

def make_tree():
    return RandomForestRegressor(**rf_params)

def make_mlp():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(32,),
            alpha=0.001,
            early_stopping=True,
            max_iter=300,
            n_iter_no_change=10,
            random_state=SEED,
        ),
    )

print(
    "Train / valid scaffold groups:",
    len(np.unique(train_scaffolds)),
    len(np.unique(valid_scaffolds)),
)
print("GroupKFold holdout sizes:", [len(hold) for _, hold in inner_splits])


Train / valid scaffold groups: 59 97
GroupKFold holdout sizes: [317, 254, 110, 110, 111]


## Steps

### 1. 拟合独立基础模型并记录收敛警告

独立模型为单模型和简单平均提供对照。警告被捕获并展示，不做全局忽略。


In [3]:
tree = make_tree()
mlp = make_mlp()
stack = StackingRegressor(
    estimators=[("tree", tree), ("mlp", mlp)],
    final_estimator=Ridge(alpha=1.0),
    cv=inner_splits,
    passthrough=False,
    n_jobs=1,
)

fitted_tree = clone(tree).fit(X_train, y_train)
with warnings.catch_warnings(record=True) as mlp_caught:
    warnings.simplefilter("always", ConvergenceWarning)
    fitted_mlp = clone(mlp).fit(X_train, y_train)

tree_pred = fitted_tree.predict(X_valid)
mlp_pred = fitted_mlp.predict(X_valid)
mean_pred = (tree_pred + mlp_pred) / 2


### 2. 只在外部训练集拟合 scaffold-aware stack

split 列表由 `GroupKFold` 在 `X_train` 内部生成；每折的 scaffold 集合互斥。`StackingRegressor` 再用全部训练数据重拟合基础模型用于外部预测。


In [4]:
started = perf_counter()
with warnings.catch_warnings(record=True) as stack_caught:
    warnings.simplefilter("always", ConvergenceWarning)
    stack.fit(X_train, y_train)
stack_seconds = perf_counter() - started
stack_pred = stack.predict(X_valid)

warning_rows = []
for component, caught in [
    ("mlp_only", mlp_caught),
    ("oof_stack", stack_caught),
]:
    messages = [
        str(item.message)
        for item in caught
        if issubclass(item.category, ConvergenceWarning)
    ]
    warning_rows.append({
        "component": component,
        "convergence_warning_count": len(messages),
        "messages": " | ".join(messages[:3]) or "none",
    })
warning_report = pd.DataFrame(warning_rows)
display(warning_report)
print("Stack fit seconds:", round(stack_seconds, 3))


,component,convergence_warning_count,messages
0,mlp_only,0,none
1,oof_stack,0,none


Stack fit seconds: 3.283


### 3. 统一评价


In [5]:
predictions = {
    "tree_only": tree_pred,
    "mlp_only": mlp_pred,
    "simple_mean": mean_pred,
    "oof_stack": stack_pred,
}
rows = []
for name, prediction in predictions.items():
    rows.append({
        "variant": name,
        "rmse": root_mean_squared_error(y_valid, prediction),
        "mae": mean_absolute_error(y_valid, prediction),
        "r2": r2_score(y_valid, prediction),
    })
results = (
    pd.DataFrame(rows)
    .sort_values("rmse")
    .reset_index(drop=True)
)
display(results.round(4))


,variant,rmse,mae,r2
0,tree_only,1.7031,1.3124,0.2578
1,oof_stack,1.8990,1.4666,0.0772
2,simple_mean,2.1692,1.8014,-0.2041
3,mlp_only,3.0735,2.5418,-1.4173


## Checks

检查禁止 `prefit`、关闭原始特征直通、OOF 覆盖完整，并同时断言外部和每个内部折都没有 scaffold 重叠。


In [6]:
assert stack.cv != "prefit"
assert stack.passthrough is False
assert set(train_scaffolds).isdisjoint(set(valid_scaffolds))
assert np.all(coverage == 1)
for fit_idx, hold_idx in inner_splits:
    assert set(train_scaffolds[fit_idx]).isdisjoint(
        set(train_scaffolds[hold_idx])
    )
assert all(len(pred) == len(y_valid) for pred in predictions.values())
assert np.isfinite(results[["rmse", "mae", "r2"]]).all().all()
assert warning_report["convergence_warning_count"].ge(0).all()
print("Scaffold-aware OOF checks passed; test remains sealed.")


Scaffold-aware OOF checks passed; test remains sealed.


## Next Steps

不因一次排名宣布 stacking 有效。Day 24 保持同一模型配方和 scaffold split，加入真正的 `stack_tree_only` 与 `stack_tree_mlp` 消融；若简单平均同样好，应优先考虑透明方案。
